# Reproduce the structured cohort counts
Designed synthetic coverage; not real student choices or independent validity. Holdouts stay separate. See the linked method report for binding limitations.

In [ ]:
from pathlib import Path
import json, gzip, hashlib
from collections import Counter
root = Path.cwd()
if not (root / "dist").exists():
    root = root.parents[1]  # run from research/cohort-056
manifest = json.loads((root / "dist/developer/current-cohort-data.json").read_text())
rows = []
for part in manifest["parts"]:
    raw = (root / "dist/developer" / part["url"]).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == part["sha256"]
    data = json.loads(gzip.decompress(raw))
    assert data["runId"] == manifest["runId"]
    rows.extend(data["rows"])
assert len(rows) == len({r["id"] for r in rows}) == 1012
base = [r for r in rows if r["phase"] != "diagnostic049"]
assert len(base) == 1000
print(manifest["simulation"])
for phase, n in Counter(r["phase"] for r in rows).items():
    group = [r for r in rows if r["phase"] == phase]
    print(phase, n, "any Top", sum(bool(r["top"]) for r in group), "degree Top", sum(any(p["degree"] for p in r["top"]) for r in group))
